In [0]:
import pyspark.sql.functions as F
from delta.tables import DeltaTable
df=spark.read.table("retaildataplatform.bronze.sqlserver_customers")
def transform_column(df):
    df=df.withColumn('customer_name',F.lower(df['customer_name']))\
        .withColumn('name_parts',F.split(F.col('customer_name'),","))\
        .withColumn('Firs_name',F.expr("get(name_parts,1)"))\
        .withColumn('Last_name',F.expr("get(name_parts,0)"))\
        .withColumn('postcode', F.regexp_replace(F.col('postcode'), r'\.0$', ''))\
        .withColumn('city',F.upper(F.col('city')))\
        .withColumn('valid_from',F.from_unixtime('valid_from').cast('timestamp'))\
        .withColumn('valid_to',F.from_unixtime(F.expr("try_cast(valid_to as long)")).cast('timestamp'))\
        .withColumn('country',F.lit('USA'))

    df=df.drop('name_parts','customer_name','file_path')
    df=df.dropDuplicates(['customer_id'])

    return df.select(
    'customer_id',
    'Firs_name',
    'Last_name',
    'tax_id',
    'tax_code',
    'state',
    'city',
    'country',
    'postcode',
    'street',
    'number',
    'unit',
    'region',
    'district',
    'lon',
    'lat',
    'ship_to_address',
    'valid_from',
    'valid_to',
    'units_purchased',
    'loyalty_segment',
    'last_updated_ts',
    )

df=spark.read.table("retaildataplatform.bronze.sqlserver_customers")
cleaned_customer=transform_column(df)
cleaned_customer.display()

target_table="retaildataplatform.silver.customers"

if spark.catalog.tableExists("retaildataplatform.silver.customers"):
    print('Table Exists proceed with SCD-1 process.')
    # Load Target
    target=DeltaTable.forName(spark,target_table) # Use this when your Delta table is registered in the metastore (like Hive or Unity Catalog).
    # SCD Type-1
    # Match in source and target --> Key exist in target
    (
    target.alias("target").merge(cleaned_customer.alias('source'),"target.customer_id=source.customer_id")

    # Update-> When records are Matching
    .whenMatchedUpdateAll()

    # Insert-> When records are not Matching
    .whenNotMatchedInsertAll()

    .execute()
        
    )
else:
    print('Table does not exist- First create Schema(Table)')
    spark.sql("create schema if not exists retaildataplatform.silver")
    cleaned_customer.write.format('delta').saveAsTable('retaildataplatform.silver.customers')